# 02 - Baseline clásico: Lexicones + SVM

Este notebook implementa el baseline clásico:
1. Extraer aspectos con reglas + POS tagging (spaCy).
2. Etiquetar sentimientos por aspecto usando un lexicón.
3. Entrenar un clasificador SVM TF-IDF con auxiliary sentence.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from sklearn.model_selection import train_test_split

from src.aspects.extractor import extract_aspects, load_spacy_model
from src.classical.lexicon import DEFAULT_LEXICON, load_lexicon, score_aspect_lexicon
from src.classical.svm_classifier import SVMAspectClassifier
from src.data.loader import load_reviews
from src.evaluation.metrics import evaluate_predictions

## 1. Carga de datos y modelo spaCy

In [ ]:
df = load_reviews("../data/sample/reviews_sample.csv")
nlp = load_spacy_model()
lexicon = load_lexicon()
print(f"Reseñas: {len(df)} | Lexicón: {len(lexicon)} términos")

## 2. Extracción de aspectos

Usamos POS tagging para sacar sustantivos relevantes.

In [ ]:
for _, row in df.head(3).iterrows():
    print('Reseña :', row['text'])
    print('Aspectos:', extract_aspects(row['text'], nlp))
    print('---')

## 3. Scoring por lexicón

Aplicamos `score_aspect_lexicon` con ventana de 5 palabras.

In [ ]:
sample = df.iloc[0]['text']
for asp in extract_aspects(sample, nlp):
    print(f"{asp:>15} -> {score_aspect_lexicon(sample, asp, lexicon):+.2f}")

## 4. Construcción del dataset débilmente supervisado

In [ ]:
def to_label(score: float) -> str:
    if score > 0.1:
        return 'pos'
    if score < -0.1:
        return 'neg'
    return 'neu'

texts, aspects, labels = [], [], []
for _, row in df.iterrows():
    text = row['text']
    for asp in extract_aspects(text, nlp):
        texts.append(text)
        aspects.append(asp)
        labels.append(to_label(score_aspect_lexicon(text, asp, lexicon)))
print(f"Total ejemplos: {len(texts)}")

## 5. Entrenamiento SVM

In [ ]:
t_tr, t_te, a_tr, a_te, y_tr, y_te = train_test_split(
    texts, aspects, labels, test_size=0.2, random_state=42, stratify=labels if min(labels.count(c) for c in set(labels)) >= 2 else None)

clf = SVMAspectClassifier()
clf.fit(t_tr, a_tr, y_tr)
preds = clf.predict(t_te, a_te).tolist()
metrics = evaluate_predictions(y_te, preds)
metrics

## 6. Conclusiones
- El lexicón sólo cubre términos comunes; aspectos sin palabras sentimentales cercanas devuelven 0.
- El SVM mejora sobre el lexicón si hay suficientes ejemplos.
- Comparar con BETO (notebook 03) y LLM (notebook 04).